# Лабораторная работа 2.4

Решение системы $A x = b$ методом наименьших квадратов с помощью $QR$-разложения. Полное условие [здесь](direct_methods.md).

In [29]:
import math

import numpy as np


np.set_printoptions(precision=6, suppress=True)


def compute_vector_inner_product(
    lhs: np.ndarray,
    rhs: np.ndarray,
) -> float:
    """Вычисляет евклидово скалярное произведение двух векторов."""
    if lhs.shape != rhs.shape:
        raise ValueError("Векторы должны иметь одинаковую размерность.")

    inner_product = 0.0
    for i in range(lhs.size):
        inner_product += float(lhs[i]) * float(rhs[i])
    return inner_product


def compute_vector_two_norm(v: np.ndarray) -> float:
    """Вычисляет евклидову норму вектора."""
    return math.sqrt(compute_vector_inner_product(v, v))


def validate_QR_solver_input(
    A_matrix: np.ndarray,
    b_vector: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Проверяет корректность входных данных."""
    matrix_array = np.asarray(A_matrix, dtype=float)
    if matrix_array.ndim != 2:
        raise ValueError("A должна быть матрицей.")

    row_count, column_count = matrix_array.shape
    if row_count < column_count:
        raise ValueError("Для матрицы A должно выполняться условие m >= n.")

    vector_array = np.asarray(b_vector, dtype=float)
    if vector_array.ndim == 2:
        if 1 not in vector_array.shape:
            raise ValueError("b должен быть вектором-столбцом.")
        vector_array = vector_array.reshape(-1)
    elif vector_array.ndim != 1:
        raise ValueError("b должен быть вектором.")

    if vector_array.size != row_count:
        raise ValueError("Размеры матрицы A и вектора b несовместимы.")

    return matrix_array.copy(), vector_array.copy()


def compute_QR_decomposition_modified_gram_schmidt(
    A_matrix: np.ndarray,
    tolerance: float = 1e-12,
) -> tuple[np.ndarray, np.ndarray]:
    """Находит разложение A = QR методом модифицированного Грамма-Шмидта, где Q — ортогональная, R — верхнетреугольная."""
    row_count, column_count = A_matrix.shape
    orthonormalized_columns = A_matrix.copy()
    Q_matrix = np.zeros((row_count, column_count), dtype=float)
    R_matrix = np.zeros((column_count, column_count), dtype=float)

    for column_index in range(column_count):
        current_norm = compute_vector_two_norm(
            orthonormalized_columns[:, column_index],
        )
        if current_norm <= tolerance:
            raise ValueError(
                "Столбцы матрицы A должны быть линейно независимы.",
            )

        R_matrix[column_index, column_index] = current_norm
        Q_matrix[:, column_index] = (
            orthonormalized_columns[:, column_index] / current_norm
        )

        for next_column_index in range(column_index + 1, column_count):
            projection_coefficient = compute_vector_inner_product(
                Q_matrix[:, column_index],
                orthonormalized_columns[:, next_column_index],
            )
            R_matrix[column_index, next_column_index] = projection_coefficient
            orthonormalized_columns[:, next_column_index] = (
                orthonormalized_columns[:, next_column_index]
                - projection_coefficient * Q_matrix[:, column_index]
            )

    return Q_matrix, R_matrix


def solve_upper_triangular_system(
    R_matrix: np.ndarray,
    b_vector: np.ndarray,
    tolerance: float = 1e-12,
) -> np.ndarray:
    """Решает систему Rx = b, где R — верхнетреугольная матрица."""
    row_count, column_count = R_matrix.shape
    if row_count != column_count:
        raise ValueError("Матрица R должна быть квадратной.")
    if b_vector.size != row_count:
        raise ValueError("Размеры матрицы R и вектора b несовместимы.")

    solution = np.zeros(row_count, dtype=float)

    for row_index in range(row_count - 1, -1, -1):
        diagonal_element = R_matrix[row_index, row_index]
        if abs(diagonal_element) <= tolerance:
            raise ValueError("Матрица R не должна быть R вырождена.")

        accumulated_sum = 0.0
        for column_index in range(row_index + 1, row_count):
            accumulated_sum += (
                R_matrix[row_index, column_index] * solution[column_index]
            )

        solution[row_index] = (b_vector[row_index] - accumulated_sum) / diagonal_element

    return solution


def solve_least_squares_with_QR_decomposition(
    A_matrix: np.ndarray,
    b_vector: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Решает задачу Ax = b методом наименьших квадратов через QR-разложение."""
    A_copy, b_copy = validate_QR_solver_input(
        A_matrix,
        b_vector,
    )
    Q_matrix, R_matrix = compute_QR_decomposition_modified_gram_schmidt(
        A_copy,
    )
    solution = solve_upper_triangular_system(
        R_matrix,
        Q_matrix.T @ b_copy,
    )
    return Q_matrix, R_matrix, solution


def compute_solution_difference_norm(
    first_solution: np.ndarray,
    second_solution: np.ndarray,
) -> float:
    """Вычисляет евклидову норму разности двух решений."""
    difference = np.asarray(first_solution, dtype=float) - np.asarray(
        second_solution,
        dtype=float,
    )
    return compute_vector_two_norm(difference)

In [30]:
square_A_matrix = np.array(
    [
        [3.0, 1.0, 1.0],
        [1.0, 4.0, 2.0],
        [2.0, 1.0, 5.0],
    ],
    dtype=float,
)
square_b_vector = np.array([5.0, 7.0, 8.0], dtype=float)

square_Q_matrix, square_R_matrix, square_solution = (
    solve_least_squares_with_QR_decomposition(
        square_A_matrix,
        square_b_vector,
    )
)
square_numpy_solution = np.linalg.solve(
    square_A_matrix,
    square_b_vector,
)
square_solution_difference_norm = compute_solution_difference_norm(
    square_solution,
    square_numpy_solution,
)

print("a) квадратная невырожденная матрица")
print("Матрица Q =")
print(square_Q_matrix)
print("Матрица R =")
print(square_R_matrix)
print("Полученное решение x_custom =", square_solution)
print("Решение NumPy x_numpy =", square_numpy_solution)
print("Норма разности ||x_custom - x_numpy||_2 = " f"{square_solution_difference_norm}")
print()

rectangular_A_matrix = np.array(
    [
        [1.0, 0.0],
        [1.0, 1.0],
        [1.0, 2.0],
        [1.0, 3.0],
    ],
    dtype=float,
)
rectangular_b_vector = np.array([1.0, 2.0, 2.0, 4.0], dtype=float)

rectangular_Q_matrix, rectangular_R_matrix, rectangular_solution = (
    solve_least_squares_with_QR_decomposition(
        rectangular_A_matrix,
        rectangular_b_vector,
    )
)
rectangular_numpy_solution = np.linalg.lstsq(
    rectangular_A_matrix,
    rectangular_b_vector,
    rcond=None,
)[0]
rectangular_solution_difference_norm = compute_solution_difference_norm(
    rectangular_solution,
    rectangular_numpy_solution,
)

print("b) прямоугольная матрица при m > n с линейно независимыми столбцами")
print("Матрица Q =")
print(rectangular_Q_matrix)
print("Матрица R =")
print(rectangular_R_matrix)
print("Полученное решение x_custom =", rectangular_solution)
print("Решение NumPy x_numpy =", rectangular_numpy_solution)
print(
    "Норма разности ||x_custom - x_numpy||_2 = "
    f"{rectangular_solution_difference_norm}"
)

a) квадратная невырожденная матрица
Матрица Q =
[[ 0.801784 -0.265694 -0.535303]
 [ 0.267261  0.960585 -0.076472]
 [ 0.534522 -0.081752  0.841191]]
Матрица R =
[[3.741657 2.405351 4.008919]
 [0.       3.494894 1.246717]
 [0.       0.       3.517708]]
Полученное решение x_custom = [1. 1. 1.]
Решение NumPy x_numpy = [1. 1. 1.]
Норма разности ||x_custom - x_numpy||_2 = 6.280369834735101e-16

b) прямоугольная матрица при m > n с линейно независимыми столбцами
Матрица Q =
[[ 0.5      -0.67082 ]
 [ 0.5      -0.223607]
 [ 0.5       0.223607]
 [ 0.5       0.67082 ]]
Матрица R =
[[2.       3.      ]
 [0.       2.236068]]
Полученное решение x_custom = [0.9 0.9]
Решение NumPy x_numpy = [0.9 0.9]
Норма разности ||x_custom - x_numpy||_2 = 6.753223014464259e-16
